In [1]:
import json
import glob
import re
import pandas as pd
from pathlib import Path
from natsort import natsorted
import os
from sklearn.metrics import accuracy_score
from tqdm.auto import tqdm

/home/mharoon/miniconda3/envs/llm/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
RUN_LABEL = 'final_experiments_fixed'
DATASETS = os.listdir(Path('../results/') / RUN_LABEL)
RESULT_DIRS = [
    Path('../results/') / RUN_LABEL / dataset for dataset in DATASETS
]
LABELS = ['Liberal', 'Neutral', 'Conservative']

In [3]:
results = []
for result_dir in RESULT_DIRS:
    results += glob.glob(str(result_dir / '**/balanced_bertscore/*_cands*/**/test-3000.json'), recursive=True)
    results += glob.glob(str(result_dir /  '**/random/*_cands*/**/test-3000.json'), recursive=True)

In [4]:
def sanitize_prediction(x):
    x = x.split('\n')[0].lower()
    if 'neutral' in x:
        return 'Neutral'
    if 'liberal' in x:
        return 'Liberal'
    if 'conservative' in x:
        return 'Conservative'
    return ''

In [ ]:
# def format_result(model, dataset, shots, y_true, y_pred, labels):
#     suffix = ['T']
#     if 'CHANNEL' in dataset or 'SOURCE' in dataset:
#         suffix.append('S')
#     if 'DESCRIPTION' in dataset:
#         suffix.append('D')

#     ds = ','.join(suffix)
#     acc = accuracy_score(y_true, y_pred)
#     precision, recall, _, _ = precision_recall_fscore_support(y_true, y_pred, labels=labels)
#     nums = [acc] + precision.tolist() + recall.tolist()
#     nums = ['%.2f' % i  for i in nums]
#     return (' & '.join([model, ds, shots] + nums))

In [6]:
def get_suffix(dataset):
    suffix = ['T']
    if 'CHANNEL' in dataset or 'SOURCE' in dataset:
        suffix.append('S')
    if 'DESCRIPTION' in dataset:
        suffix.append('D')
    return suffix

In [7]:
compiled_results = []

In [8]:
llm_names = set()
num_shots_ = set()
datasets = set()

test_set = pd.concat([
    pd.read_json('../data/classification/news_ideology/train.json'),
    pd.read_json('../data/classification/news_ideology/test.json'),
])
test_set = test_set.set_index('article_id')
test_set['true'] = test_set['label'].map(lambda x : LABELS[x])

keys = []

for result in results:
    if '/NEWS' not in result:
        continue

    regex = f'../results/{RUN_LABEL}/(?P<dataset>.*?)/test/(?P<num_shots>[0-9]+)?_shots/.*?/[0-9]+?_cands.*?/s0/(?P<llm_name>.*)?/test-3000.json'
    groups = re.search(
        re.compile(regex),
        result
    )

    llm_name = groups.group('llm_name')
    num_shots = groups.group('num_shots')
    dataset = groups.group('dataset')

    llm_names.add(llm_name)
    num_shots_.add(num_shots)
    datasets.add(dataset)

    with open(result) as f:
        js = json.load(f)

    key = f'{llm_name}_{num_shots}_{dataset}'
    keys.append(key)
    rows = []
    for res in js['results']:
        try:
            pred = sanitize_prediction(res['pred'])
            idx = res['article_id']
            rows.append({
                key: pred,
                'idx': idx
            })
        except Exception as e:
            print(e)
            continue
    df = pd.DataFrame(rows).set_index('idx')
    df = test_set.merge(df, left_index=True, right_index=True)
    compiled_results.append(dict(
        llm_name=llm_name,
        dataset=dataset,
        num_shots=num_shots,
        acc=accuracy_score(df['true'], df[key])
    ))
    # test_set = test_set.merge(df, left_index=True, right_index=True)

keys = natsorted(keys)

In [9]:
# for dataset in sorted(datasets, key=len):
#     for llm_name in sorted(llm_names):
#         for num_shots in natsorted(num_shots_):
#             key = f'{llm_name}_{num_shots}_{dataset}'
#             compiled_results.append(dict(
#             ))
#             # print(format_result(llm_name, dataset, num_shots, test_set['true'], test_set[key], LABELS), end='\\\\\n')

In [10]:
# for dataset in sorted(datasets, key=len):
#     print('\\multicolumn{9}{c}{%s}\\\\ \\midrule' % dataset)
#     for llm_name in sorted(llm_names):
#         for num_shots in natsorted(num_shots_):
#             key = f'{llm_name}_{num_shots}_{dataset}'
#             if key in test_set.columns:
#                 print(format_result(llm_name, dataset, num_shots, test_set['true'], test_set[key], LABELS), end='\\\\\n')

In [11]:
llm_names = set()
num_shots_ = set()
datasets = set()

test_set = pd.concat([
    pd.read_json('../data/classification/yt_ideology/train.json'),
    pd.read_json('../data/classification/yt_ideology/test.json'),
])
test_set = test_set.set_index('index')
test_set['true'] = test_set['label'].map(lambda x : LABELS[x])

keys = []

for result in tqdm(results):
    if 'YT' not in result:
        continue

    regex = f'../results/{RUN_LABEL}/(?P<dataset>.*?)/test/(?P<num_shots>[0-9]+)?_shots/.*?/[0-9]+?_cands.*?/s0/(?P<llm_name>.*)?/test-3000.json'
    groups = re.search(
        re.compile(regex),
        result
    )

    llm_name = groups.group('llm_name')
    num_shots = groups.group('num_shots')
    dataset = groups.group('dataset')

    llm_names.add(llm_name)
    num_shots_.add(num_shots)
    datasets.add(dataset)

    with open(result) as f:
        js = json.load(f)

    key = f'{llm_name}_{num_shots}_{dataset}'
    keys.append(key)
    
    rows = []
    for res in js['results']:
        try:
            pred = sanitize_prediction(res['pred'])
            idx = res['index']
            rows.append({
                key: pred,
                'idx': idx
            })
        except Exception as e:
            print(e)
            continue
    df = pd.DataFrame(rows).set_index('idx')
    df = test_set.merge(df, left_index=True, right_index=True)
    compiled_results.append(dict(
        llm_name=llm_name,
        dataset=dataset,
        num_shots=num_shots,
        acc=accuracy_score(df['true'], df[key])
    ))

keys = natsorted(keys)

100%|██████████| 70/70 [00:00<00:00, 76.90it/s] 


In [12]:
test_set

,video_id,title,uploader,description,label,true
index,,,,,,
0,eXGEictCR8k,The Cost of Campaigns | Retro Report | The New...,The New York Times,The Watergate campaign finance scandals led to...,0,Liberal
1,g1BbswU3i10,U.S. States and the China Competition: Secreta...,U.S. Department of State,Secretary of State Michael R. Pompeo remarks t...,2,Conservative
2,DYt9gqzwu74,Gen. McCaffrey: Ukraine Has Access To U.S. Ant...,MSNBC,MSNBC Military Analyst Gen. Barry McCaffrey sa...,0,Liberal
3,RulUtRQHPrk,"'Speech is protected, actions aren't' - Heathe...",CNN,CNN's John Berman speaks with Susan Bro about ...,0,Liberal
4,xCFa1NkkRBQ,Mike Pence Rebukes Trump & The Metaverse Respo...,The Daily Show with Trevor Noah,Meta’s metaverse combats sexual harassment wit...,0,Liberal
...,...,...,...,...,...,...
9306,L6209AFs0ks,"Ben Carson HUMILIATES AOC With An Epic Speech,...",Real Patriots,Watch ben carson humiliate AOC and the entire ...,2,Conservative
7631,RKseZzSL2jM,Mosul: Fight against ISIS from the sky in 360 ...,BBC News,Extraordinary BBC footage allows you to join t...,1,Neutral
47486,dIW4QnAMxHM,WATCH LIVE: Biden speaks in Pittsburgh to prom...,PBS NewsHour,Stream your PBS favorites with the PBS app: ht...,0,Liberal


In [13]:
# for dataset in sorted(datasets, key=len):
#     print('\\multicolumn{9}{c}{%s}\\\\ \\midrule' % dataset)
#     for llm_name in sorted(llm_names):
#         for num_shots in natsorted(num_shots_):
#             key = f'{llm_name}_{num_shots}_{dataset}'
#             if key in test_set.columns:
#                 print(format_result(llm_name, dataset, num_shots, test_set['true'], test_set[key], LABELS), end='\\\\\n')

In [14]:
# for dataset in sorted(datasets, key=len):
#     for llm_name in sorted(llm_names):
#         for num_shots in natsorted(num_shots_):
#             key = f'{llm_name}_{num_shots}_{dataset}'
#             compiled_results.append(dict(
#                 llm_name=llm_name,
#                 dataset=dataset,
#                 num_shots=num_shots,
#                 acc=accuracy_score(test_set['true'], test_set[key])
#             ))
#             # print(format_result(llm_name, dataset, num_shots, test_set['true'], test_set[key], LABELS), end='\\\\\n')

In [ ]:
llm_names = set()
num_shots_ = set()
datasets = set()

test_set = pd.concat([
    pd.read_json('../data/classification/news_baly/train.json'),
    pd.read_json('../data/classification/news_baly/test.json'),
])
test_set = test_set.set_index('ID')
test_set['true'] = test_set['label'].map(lambda x : LABELS[x])

keys = []

for result in results:
    if 'BALYNEWS' not in result:
        continue

    regex = f'../results/{RUN_LABEL}/(?P<dataset>.*?)/test/(?P<num_shots>[0-9]+)?_shots/.*?/[0-9]+?_cands.*?/s0/(?P<llm_name>.*)?/test-3000.json'
    groups = re.search(
        re.compile(regex),
        result
    )

    llm_name = groups.group('llm_name')
    num_shots = groups.group('num_shots')
    dataset = groups.group('dataset')

    llm_names.add(llm_name)
    num_shots_.add(num_shots)
    datasets.add(dataset)

    with open(result) as f:
        js = json.load(f)

    key = f'{llm_name}_{num_shots}_{dataset}'
    keys.append(key)
    rows = []
    for res in js['results']:
        try:
            pred = sanitize_prediction(res['pred'])
            idx = res['ID']
            rows.append({
                key: pred,
                'idx': idx
            })
        except Exception as e:
            print(e)
            continue
    df = pd.DataFrame(rows).set_index('idx')
    df = test_set.merge(df, left_index=True, right_index=True)
    compiled_results.append(dict(
        llm_name=llm_name,
        dataset=dataset,
        num_shots=num_shots,
        acc=accuracy_score(df['true'], df[key])
    ))
    # test_set = test_set.merge(df, left_index=True, right_index=True)

keys = natsorted(keys)

In [23]:
key

'GPT4_4_BALYNEWSIDEOLOGY'

In [ ]:
compiled_results = []
for dataset in sorted(datasets, key=len):
    for llm_name in sorted(llm_names):
        for num_shots in natsorted(num_shots_):
            key = f'{llm_name}_{num_shots}_{dataset}'
            # compiled_results.append(dict(
            #     llm_name=llm_name,
            #     dataset=dataset,
            #     num_shots=num_shots,
            #     acc=accuracy_score(test_set['true'], test_set[key])
            # ))
            print(format_result(llm_name, dataset, num_shots, test_set['true'], test_set[key], LABELS), end='\\\\\n')

KeyError: 'GPT4_4_BALYNEWSIDEOLOGY'

In [22]:
compiled_results = pd.DataFrame(compiled_results)
compiled_results.shape

(0, 0)

In [18]:
compiled_results['dataset'].unique().tolist()

['NEWSSOURCEIDEOLOGY',
 'NEWSDESCRIPTIONIDEOLOGY',
 'NEWSSOURCEDESCRIPTIONIDEOLOGY',
 'NEWSIDEOLOGY',
 'YTCHANNELDESCRIPTIONIDEOLOGY',
 'YTIDEOLOGY',
 'YTDESCRIPTIONIDEOLOGY',
 'YTCHANNELIDEOLOGY',
 'BALYNEWSSOURCEDESCRIPTIONIDEOLOGY',
 'BALYNEWSSOURCEIDEOLOGY',
 'BALYNEWSDESCRIPTIONIDEOLOGY',
 'BALYNEWSIDEOLOGY']

In [19]:
[ 
    'NEWSIDEOLOGY',
    'NEWSSOURCEIDEOLOGY',
    'NEWSDESCRIPTIONIDEOLOGY',
    'NEWSSOURCEDESCRIPTIONIDEOLOGY',
    'YTIDEOLOGY',
    'YTCHANNELIDEOLOGY',
    'YTDESCRIPTIONIDEOLOGY',
    'YTCHANNELDESCRIPTIONIDEOLOGY',
    'BALYNEWSIDEOLOGY'
    'BALYNEWSSOURCEIDEOLOGY',
    'BALYNEWSDESCRIPTIONIDEOLOGY',
    'BALYNEWSSOURCEDESCRIPTIONIDEOLOGY',
]

['NEWSIDEOLOGY',
 'NEWSSOURCEIDEOLOGY',
 'NEWSDESCRIPTIONIDEOLOGY',
 'NEWSSOURCEDESCRIPTIONIDEOLOGY',
 'YTIDEOLOGY',
 'YTCHANNELIDEOLOGY',
 'YTDESCRIPTIONIDEOLOGY',
 'YTCHANNELDESCRIPTIONIDEOLOGY',
 'BALYNEWSIDEOLOGYBALYNEWSSOURCEIDEOLOGY',
 'BALYNEWSDESCRIPTIONIDEOLOGY',
 'BALYNEWSSOURCEDESCRIPTIONIDEOLOGY']